# 🌟 YıldızSezar: Turkish E-Commerce Review Classifier
Bu notebook, **ConvBERT** tabanlı çok sınıflı (1-5 Yıldız) Türkçe duygu analizi modelimi Google'ın ücretsiz GPU'su ile test etmeniz için hazırlanmıştır.

Lütfen üst menüden **Runtime -> Change runtime type** seçerek Hardware accelerator kısmını **T4 GPU** yaptığınızdan emin olun.

In [ ]:
# Gerekli kütüphaneleri kuruyoruz
!pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

# Modeli Hugging Face üzerinden indiriyoruz
model_id = "ilkayO/yildizsezar-convbert"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)
model.to(device)
model.eval()
print("Model Başarıyla Yüklendi!")

In [ ]:
# İstediğiniz müşteri yorumunu buraya yazın
musteri_yorumu = "Ürün elime çok hızlı ulaştı, paketleme harikaydı. Kesinlikle tavsiye ederim!"

inputs = tokenizer(musteri_yorumu, return_tensors="pt", truncation=True, max_length=256)
inputs = {key: val.to(device) for key, val in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class_id = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class_id].item() * 100

print(f"Yorum: '{musteri_yorumu}'")
print(f"Tahmin: {predicted_class_id + 1} Yıldız")
print(f"Güven Skoru: %{confidence:.2f}")